# Figure 1B (top) & Figure S2A: Two-Arm Bandit Reversal Learning Curves

| Field | Value |
|---|---|
| **Manuscript panels / stats** | Fig 1B (top: EMU n=19 choice probability & reward contingencies), Fig S2A (online n=200 choice probability & reward contingencies), Methods EMU exclusion (d' <= 0.25 -> n=19) |
| **Inputs** | `$MEMORYBANDIT_DATA_DIR/work/qasims01/MemoryBanditData/EMU/{learn_df_RWH.csv,full_df_RWH.csv}`, `$MEMORYBANDIT_DATA_DIR/work/qasims01/MemoryBanditData/Prolific/DelayedMemoryTest/learn_df_prolific_v2.csv` |
| **Outputs** | `figures/Fig1B_EMU_reversal_curve.pdf`, `figures/FigS2A_online_reversal_curve.pdf` |
| **Dependencies** | `membandit`, `numpy`, `pandas`, `matplotlib`, `seaborn` |
| **Expected runtime** | < 10 seconds |
| **Upstream steps** | `00_prolific_preprocessing.ipynb` (for online v2 CSV) |


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from membandit import paths, set_plot_style

set_plot_style()
EMU_DIR = Path(paths.get_emu_behavior_dir())
PROLIFIC_DIR = Path(paths.get_prolific_dir())
FIG_DIR = Path(paths.figure_output_root())
FIG_DIR.mkdir(parents=True, exist_ok=True)


## Shared reversal-curve plotting helper

Plots deck-selection probabilities (`male` orange vs `female` purple) across the 80 trials of the two-arm bandit task with 95% confidence intervals, vertical reversal lines (`[16, 33, 48, 65]`), and horizontal lines indicating the reward probability (`p($) = 0.80`) of the more rewarding deck in each block.

In [ ]:
REVERSAL_TRIALS = [16, 33, 48, 65]
BLOCK_SPANS = [(0, 16, 1, "orange"), (16, 33, 2, "purple"), (33, 48, 3, "orange"), (48, 65, 4, "purple"), (65, 80, 5, "orange")]


def plot_reversal_learning_curves(
    learn_df: pd.DataFrame,
    *,
    cohort_title: str,
    out_pdf: Path,
    high_reward_prob: float = 0.80,
) -> plt.Figure:
    df = learn_df.copy()
    if "good_dm_performance" in df.columns:
        df = df.loc[df["good_dm_performance"].astype(bool)].copy()
    trial_col = "learn_trials" if "learn_trials" in df.columns else "trials"

    fig, ax = plt.subplots(1, 1, dpi=300, figsize=(6, 4))
    sns.lineplot(
        data=df, x=trial_col, y="male", color="orange", ax=ax,
        errorbar=("ci", 95), label="male deck",
    )
    sns.lineplot(
        data=df, x=trial_col, y="female", color="purple", ax=ax,
        errorbar=("ci", 95), label="female deck",
    )
    ax.vlines(REVERSAL_TRIALS, 0, 1, color="black", lw=1.0, ls="-")
    for t0, t1, blk, col in BLOCK_SPANS:
        p_val = high_reward_prob
        if "prob" in df.columns and "block" in df.columns:
            sub = df.loc[pd.to_numeric(df["block"], errors="coerce") == blk, "prob"]
            if len(sub) and np.isfinite(sub.max()) and sub.max() > 0.5:
                p_val = float(sub.max())
        ax.hlines(p_val, t0, t1, color=col, lw=1.5, ls="--")

    ax.set_xlim(0, 80)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("trials (2-arm bandit)")
    ax.set_ylabel("p(draw)")
    ax.set_title(cohort_title)
    sns.despine(ax=ax)
    fig.tight_layout()
    fig.savefig(out_pdf, dpi=300, bbox_inches="tight")
    print(f"Saved {out_pdf} (n={df['participant'].nunique()} participants)")
    return fig


## Figure 1B (top): Neurosurgical (EMU) cohort (`n = 19`)

Also verifies the Methods exclusion rule (`d' <= 0.25` on Day-1 recognition excluded; `n = 19` retained).

In [ ]:
emu_learn_path = EMU_DIR / "learn_df_RWH.csv"
emu_full_path = EMU_DIR / "full_df_RWH.csv"

if emu_learn_path.exists() and emu_full_path.exists():
    emu_learn = pd.read_csv(emu_learn_path)
    emu_full = pd.read_csv(emu_full_path)
    d1 = emu_full.loc[emu_full["condition"].astype(str).str.lower().isin(["day1", "day_1", "day 1"])]
    subj_dprime = d1.groupby("participant")["dprime"].first()
    keep_ids = subj_dprime.loc[subj_dprime > 0.25].index
    emu_learn = emu_learn.loc[emu_learn["participant"].isin(keep_ids)].copy()
    assert emu_learn["participant"].nunique() == 19, f"Expected n=19 EMU participants with d'>0.25, got {emu_learn['participant'].nunique()}"
    plot_reversal_learning_curves(
        emu_learn,
        cohort_title=f"EMU (n={emu_learn['participant'].nunique()})",
        out_pdf=FIG_DIR / "Fig1B_EMU_reversal_curve.pdf",
    )
else:
    print(f"Data not mounted at {EMU_DIR}; set MEMORYBANDIT_DATA_DIR to run.")


## Figure S2A: Online (Prolific) cohort (`n = 200`)

In [ ]:
pro_learn_path = PROLIFIC_DIR / "learn_df_prolific_v2.csv"
if pro_learn_path.exists():
    pro_learn = pd.read_csv(pro_learn_path)
    if "good_dm_performance" in pro_learn.columns:
        pro_learn = pro_learn.loc[pro_learn["good_dm_performance"].astype(bool)].copy()
    assert pro_learn["participant"].nunique() == 200, f"Expected n=200 online participants, got {pro_learn['participant'].nunique()}"
    plot_reversal_learning_curves(
        pro_learn,
        cohort_title=f"online (n={pro_learn['participant'].nunique()})",
        out_pdf=FIG_DIR / "FigS2A_online_reversal_curve.pdf",
    )
else:
    print(f"Data not mounted at {PROLIFIC_DIR}; set MEMORYBANDIT_DATA_DIR to run.")
